In [ ]:
import csv
import io
import json
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path
from tempfile import NamedTemporaryFile

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parents[0]
OUTPUT = TASK / 'output'
CONFIG = json.loads((ROOT / '.report-wizard.json').read_text(encoding='utf-8'))

In [ ]:
def repo_path(root, name):
    relative = Path(name)
    path = root / relative
    if relative.is_absolute() or '..' in relative.parts or not path.resolve().is_relative_to(root):
        raise ValueError(f'Use a path inside the repository: {name}')
    return path

def read_table(path):
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.reader(stream)
        columns = next(reader, [])
        if not columns or any(not col.strip() for col in columns) or len(set(columns)) != len(columns):
            raise ValueError(f'{path.name}: CSV needs distinct, nonempty column names')
        rows = []
        for row in reader:
            if not row:
                continue
            if len(row) != len(columns):
                raise ValueError(f'{path.name}: malformed record ending on line {reader.line_num}')
            rows.append(dict(zip(columns, row)))
    return columns, rows

def formatted_value(raw, decimals, percent):
    try:
        number = Decimal(raw.strip())
        if not number.is_finite() or (number and abs(number.adjusted()) > 100):
            raise ValueError('value must be a finite number within the supported range')
        if percent:
            number *= 100
        if not isinstance(decimals, int) or not 0 <= decimals <= 15:
            raise ValueError('decimal places must be between 0 and 15')
        return format(number, f'.{decimals}f')
    except InvalidOperation as error:
        raise ValueError(f'expected a number, got {raw!r}') from error

def evaluate(macro, columns, rows):
    fields = set(macro['where']) | {macro['column']}
    missing = fields - set(columns)
    if missing:
        raise ValueError('missing column(s): ' + ', '.join(sorted(missing)))
    matches = [row for row in rows if all(row[field] == value for field, value in macro['where'].items())]
    if len(matches) != 1:
        raise ValueError(f'filters matched {len(matches)} rows; select exactly one')
    raw = matches[0][macro['column']]
    return raw, formatted_value(raw, macro['decimals'], macro['percent'])

def link_name(task, source):
    return str(Path(task) / 'input' / source)

In [ ]:
def render_report(root, config, *, use_links=True):
    tables, records, names = {}, [], set()
    definitions = ['% generated by the reporting notebook; refresh with make report']
    slash = chr(92)
    for macro in config['macros']:
        name, source = macro['name'], macro['source']
        try:
            if not re.fullmatch('[A-Za-z]+', name) or name in names:
                raise ValueError('command names must be unique and contain only A–Z and a–z')
            if source not in tables:
                relative = link_name(config['task'], source) if use_links else source
                tables[source] = read_table(repo_path(root, relative))
            raw, value = evaluate(macro, *tables[source])
        except (ValueError, OSError, csv.Error) as error:
            raise ValueError(f'{slash}{name} in {source}: {error}') from error
        names.add(name)
        definitions.append(f'{slash}newcommand{{{slash}{name}}}{{{value}}}')
        records.append([name, source, macro['column'], json.dumps(macro['where'], ensure_ascii=False),
                        raw, value, macro['decimals'], macro['percent']])
    manifest = io.StringIO(newline='')
    writer = csv.writer(manifest, lineterminator='\n')
    writer.writerow(['macro', 'source', 'column', 'filters', 'raw', 'formatted', 'decimals', 'percent'])
    writer.writerows(records)
    return '\n'.join(definitions) + '\n', manifest.getvalue()

def write_atomic(path, data):
    if path.is_file() and not path.is_symlink() and path.read_bytes() == data:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=path.parent, delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(data)
    try:
        temporary.chmod((path.stat().st_mode & 0o777) if path.is_file() else 0o644)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)

In [ ]:
results, manifest = render_report(ROOT, CONFIG)

In [ ]:
write_atomic(OUTPUT / 'macro_manifest.csv', manifest.encode('utf-8'))
write_atomic(OUTPUT / 'results.tex', results.encode('utf-8'))